In [3]:
import pandas as pd


In [4]:
df = pd.read_excel("CorporateSalary.xlsx")
df
df.head(2)


,Designation,Education,Department,TenureYears,ProjectsCompleted,PerformanceScore,Salary
0,Senior Analyst,Bachelor's,Data Science,8.7,12,4.6,119300
1,Director,Master's,Marketing,6.9,13,4.3,212200


In [5]:
y = df['Salary']
y

X = df.drop('Salary',axis=1)
X


,Designation,Education,Department,TenureYears,ProjectsCompleted,PerformanceScore
0,Senior Analyst,Bachelor's,Data Science,8.7,12,4.6
1,Director,Master's,Marketing,6.9,13,4.3
2,Manager,Bachelor's,Engineering,3.9,14,2.7
3,Lead Consultant,Master's,Data Science,5.3,13,2.7
4,Junior Analyst,PhD,HR,3.4,15,5.0
...,...,...,...,...,...,...
115,Director,PhD,HR,0.4,1,2.3
116,Senior Manager,Master's,Data Science,0.2,0,2.1
117,Junior Analyst,Bachelor's,Engineering,0.0,1,1.9
118,Manager,Master's,Sales,0.2,2,2.0


In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train_true, y_test_true = train_test_split(X,y,test_size=0.20,random_state=42)


In [7]:
# Transformation 
Numerica_Colums = ['TenureYears','ProjectsCompleted','PerformanceScore']
Ordered_Columns = ['Designation', 'Education']
One_Hot_Columns = ['Department']

Designation_Order = ['Junior Analyst', 'Senior Analyst', 'Lead Consultant', 'Manager', 'Senior Manager','Director']
Education_Order = ["Bachelor's", "Master's", 'PhD']


In [8]:
# Transformation Actual Process
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler


OrdinalEncoding_01 = ("Oridnal_Encoder",OrdinalEncoder(categories=[Designation_Order, Education_Order]),Ordered_Columns,)
OneHotEncoding_01 =  ("One_Hot_Encoder",OneHotEncoder(sparse_output=False, drop="first", handle_unknown="ignore"),One_Hot_Columns,)
Scaling = ("Numerical_Coluns", StandardScaler(), Numerica_Colums)


transformer_engine = ColumnTransformer(transformers=[OrdinalEncoding_01,OneHotEncoding_01,Scaling]).set_output(transform="pandas")

transformer_engine



ColumnTransformer(transformers=[('Oridnal_Encoder',
                                 OrdinalEncoder(categories=[['Junior Analyst',
                                                             'Senior Analyst',
                                                             'Lead Consultant',
                                                             'Manager',
                                                             'Senior Manager',
                                                             'Director'],
                                                            ["Bachelor's",
                                                             "Master's",
                                                             'PhD']]),
                                 ['Designation', 'Education']),
                                ('One_Hot_Encoder',
                                 OneHotEncoder(drop='first',
                                               handle_unknown='ignore',
                                               sparse_output=False),
                                 ['Department']),
                                ('Numerical_Coluns', StandardScaler(),
                                 ['TenureYears', 'ProjectsCompleted',
                                  'PerformanceScore'])])

In [9]:
X_train_Transform = transformer_engine.fit_transform(X_train)

X_train_Transform.head(2)



,Oridnal_Encoder__Designation,Oridnal_Encoder__Education,One_Hot_Encoder__Department_Engineering,One_Hot_Encoder__Department_HR,One_Hot_Encoder__Department_Marketing,One_Hot_Encoder__Department_Product,One_Hot_Encoder__Department_Sales,Numerical_Coluns__TenureYears,Numerical_Coluns__ProjectsCompleted,Numerical_Coluns__PerformanceScore
42,0.0,0.0,0.0,1.0,0.0,0.0,0.0,-0.058633,1.084800,0.063193
12,3.0,2.0,0.0,0.0,0.0,1.0,0.0,-1.049288,-0.200889,-1.030770


In [10]:
import joblib

joblib.dump(transformer_engine,"transform.pkl")



['transform.pkl']

In [11]:
from sklearn.linear_model import Ridge

ridgeModel = Ridge(alpha=1.0)

ridgeModel.fit(X_train_Transform,y_train_true)




Ridge()

In [12]:
#Save the Trained Model
import joblib
joblib.dump(ridgeModel,"Ridge_Model.pkl")



['Ridge_Model.pkl']

In [ ]:
# Get y_train_Pred
Ridge_Model = joblib.load("Ridge_Model.pkl")

y_train_pred = Ridge_Model.predict(X_train_Transform)

y_train_pred



array([118309.63421913, 121676.02889321,  84464.76079444,  56503.47942935,
       176920.17625   , 114492.64116978,  54666.28421522, 133677.88116301,
       101461.88042787, 160320.11347481, 123819.86225384,  -2412.93122299,
       138473.48131203, 126383.93654492,  75428.53648328, 177917.56167631,
       177505.41990883, 139548.20042998, 269458.45614257, 128955.4448212 ,
       113306.49443822, 127845.48333634,  69066.47970731,  85575.34964931,
        69391.93313237, 140255.57887234,  94231.86080594, 145375.30615477,
        94637.4403962 ,  47882.75121348, 224429.9288533 ,  70391.7170217 ,
       179844.25649632, 173664.36452968, 139352.50644698, 112170.92697029,
       130379.37477918, 134469.84538942, 104344.02866552, 139814.17946011,
       153654.01650023, 117467.35542297,  82649.05099145, 104121.85287147,
       134583.87906086, 111989.47642577, 169590.55391917,   1361.46303664,
        79737.23963861,  90395.32645441,   2652.81777608, 118583.65052528,
       126662.54723081, 2

In [45]:
# Train Phase Evaluation
# MSE, MAE, RMSE, R2, A_R2

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

MSE = mean_squared_error(y_train_true,y_train_pred) 
MAE = mean_absolute_error(y_train_true,y_train_pred)
RMSE = np.sqrt(MSE)
R2 = r2_score(y_train_true,y_train_pred)

n= len(X_train)
p = X_train.shape[1]
A_R2 = 1-(1-R2)*(n-1)/(n-p-1)


print(round(MSE,4))
print(round(MAE,4))
print(round(RMSE,4))
print(round(R2,4))
print(round(A_R2,4))



1201188894.0604
21647.4031
34658.1721
0.7675
0.7519


In [ ]:

def linear_Metrics(x_Data,True_Value,Pred_Value):

    MSE = mean_squared_error(True_Value,Pred_Value) 
    MAE = mean_absolute_error(True_Value,Pred_Value)
    RMSE = np.sqrt(MSE)
    R2 = r2_score(True_Value,Pred_Value)

    n = len(x_Data)
    p = x_Data.shape[1]
    A_R2 = 1-(1-R2)*(n-1)/(n-p-1)

    merticsDict = {"MSE": MSE,
                   "MAE":MAE,
                   "RMSE": RMSE,
                   "R2": R2,
                   "A_R2":A_R2}
    return merticsDict

new_Value =  linear_Metrics(X_train,y_train_true,y_train_pred)

new_Value



{'MSE': 1201188894.0603888,
 'MAE': 21647.40306639072,
 'RMSE': 34658.17211077914,
 'R2': 0.767548620773734,
 'A_R2': 0.751877741275334}

In [56]:
# Testing Phase 


X_test


,Designation,Education,Department,TenureYears,ProjectsCompleted,PerformanceScore
44,Senior Analyst,Bachelor's,Marketing,3.4,10,4.2
47,Lead Consultant,Master's,HR,5.4,21,4.0
4,Junior Analyst,PhD,HR,3.4,15,5.0
55,Senior Manager,Bachelor's,Data Science,11.2,10,3.2
26,Junior Analyst,Master's,Product,6.7,6,4.0
64,Senior Analyst,Bachelor's,Product,1.9,17,3.8
73,Manager,Bachelor's,Product,9.8,24,4.0
10,Junior Analyst,Bachelor's,Marketing,1.3,18,4.4
40,Junior Analyst,PhD,Product,1.0,24,3.9
107,Director,Master's,Sales,21.3,68,5.0


In [55]:
transform_Obj = joblib.load("transform.pkl")

X_test_transform = transform_Obj.fit_transform(X_test)

X_test_transform


,Oridnal_Encoder__Designation,Oridnal_Encoder__Education,One_Hot_Encoder__Department_Engineering,One_Hot_Encoder__Department_HR,One_Hot_Encoder__Department_Marketing,One_Hot_Encoder__Department_Product,One_Hot_Encoder__Department_Sales,Numerical_Coluns__TenureYears,Numerical_Coluns__ProjectsCompleted,Numerical_Coluns__PerformanceScore
44,1.0,0.0,0.0,0.0,1.0,0.0,0.0,-0.609075,-0.778328,0.228976
47,2.0,1.0,0.0,1.0,0.0,0.0,0.0,-0.234260,-0.047087,-0.068074
4,0.0,2.0,0.0,1.0,0.0,0.0,0.0,-0.609075,-0.445946,1.417176
55,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.852705,-0.778328,-1.256274
26,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.009370,-1.044233,-0.068074
64,1.0,0.0,0.0,0.0,0.0,1.0,0.0,-0.890186,-0.312993,-0.365124
73,3.0,0.0,0.0,0.0,0.0,1.0,0.0,0.590334,0.152342,-0.068074
10,0.0,0.0,0.0,0.0,1.0,0.0,0.0,-1.002631,-0.246517,0.526026
40,0.0,2.0,0.0,0.0,0.0,1.0,0.0,-1.058853,0.152342,-0.216599
107,5.0,1.0,0.0,0.0,0.0,0.0,1.0,2.745521,3.077303,1.417176


In [57]:
y_test_pred = Ridge_Model.predict(X_test_transform)

y_test_pred


array([ 73445.7162719 , 124936.20642306,  67811.72609133, 174569.05863553,
        53684.78998544,  73251.75601633, 173593.71972775,  59759.3769694 ,
        68686.14129615, 377448.05707092, 106411.91050879, 163039.67168551,
       157870.11985788,  98710.93957121,  91444.46013398, 139263.07408404,
       167968.52207696, 109738.3568659 , 179219.82318105, 293069.87418423,
       116636.46413114,  95467.65923478,  71670.18007235, 142858.77123046])

In [58]:
# Testing Phase 
linear_Metrics(X_test,y_test_true,y_test_pred)



{'MSE': 5972549592.996896,
 'MAE': 40829.60031335128,
 'RMSE': 77282.27217801567,
 'R2': 0.6485629827686991,
 'A_R2': 0.5245263884517692}